# Evaluation — Faithfulness, Deletion Curves, and Missing-Modality Robustness

[![GitHub](https://img.shields.io/badge/GitHub-manasdutta04%2Fmultimodal--fake--news--detector-181717?logo=github&logoColor=white)](https://github.com/manasdutta04/multimodal-fake-news-detector)

This notebook is Module 04 of *Multimodal Fake News Detector with Explainability*. It does **not** train a new classifier. It verifies that explanations from Modules 01–03 are faithful, and measures how fusion behaves when a modality is missing.

> **Note.** Research prototype evaluation — not a production fact-check audit.

**What you get**
1. Text deletion curves (mask top SHAP tokens → confidence drop on TF-IDF+LogReg)
2. Image deletion curves (blank top Grad-CAM region → confidence drop on ResNet50)
3. Missing-modality ablation on late / cross-attn fusion (zero text or zero image embedding)
4. One comparison table: unimodal + fusion metrics + faithfulness scores

**Prerequisites on Drive (`DATA_DIR`)**
- TSVs + `image_cache/`
- `checkpoints/module01_distilbert/`
- `checkpoints/module02_resnet50/model.pt`
- `checkpoints/module03_fusion/fusion.pt`
- Optional: `module01_metrics.csv`, `module02_metrics.csv`, `module03_metrics.csv`, `module03_embeddings/`


## Environment

GPU helps for Grad-CAM / ResNet scoring. TF-IDF SHAP and fusion ablation are light.


## Dependencies


In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib seaborn pillow tqdm shap
%pip install -q "transformers>=4.40" "grad-cam>=1.5.0"

# PyTorch / torchvision ship with Colab/Kaggle GPU images.


> **Note.** After installing on a fresh runtime, restart once, then continue from imports.


In [ ]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter, ImageDraw
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet50

from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


## Paths

Same Drive folder as Modules 01–03 (`/content/drive/MyDrive/dataset`).


In [ ]:
USE_GOOGLE_DRIVE = True
# Set False and point DATA_DIR elsewhere if Drive keeps failing
FORCE_REMOUNT = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    import os

    # Reuse an existing mount if Colab already attached Drive
    if os.path.isdir("/content/drive/MyDrive") and not FORCE_REMOUNT:
        print("Drive already available at /content/drive")
    else:
        try:
            drive.mount("/content/drive", force_remount=FORCE_REMOUNT)
        except ValueError as e:
            print("Mount failed once; retrying with force_remount=True …")
            print("If this fails again: Runtime → Disconnect and delete runtime, then reconnect.")
            drive.mount("/content/drive", force_remount=True)

    DATA_DIR = "/content/drive/MyDrive/dataset"
else:
    # Fallback: zip your dataset folder to Colab Files, unzip to /content/dataset
    DATA_DIR = "/content/dataset"

TRAIN_PATH = os.path.join(DATA_DIR, "multimodal_train.tsv")
VAL_PATH = os.path.join(DATA_DIR, "multimodal_validate.tsv")
TEST_PATH = os.path.join(DATA_DIR, "multimodal_test_public.tsv")
IMAGE_CACHE = os.path.join(DATA_DIR, "image_cache")

TEXT_CKPT = os.path.join(DATA_DIR, "checkpoints", "module01_distilbert")
IMAGE_CKPT = os.path.join(DATA_DIR, "checkpoints", "module02_resnet50", "model.pt")
FUSION_CKPT = os.path.join(DATA_DIR, "checkpoints", "module03_fusion", "fusion.pt")
EMB_CACHE = os.path.join(DATA_DIR, "checkpoints", "module03_embeddings")
OUT_DIR = os.path.join(DATA_DIR, "checkpoints", "module04_evaluation")
os.makedirs(OUT_DIR, exist_ok=True)

print("DATA_DIR =", DATA_DIR)
print("MyDrive visible:", os.path.isdir("/content/drive/MyDrive"))

for p in [TRAIN_PATH, TEST_PATH, IMAGE_CACHE, TEXT_CKPT, IMAGE_CKPT, FUSION_CKPT]:
    assert os.path.exists(p), f"Missing: {p}"
print("Module 01–03 artifacts found.")


## Evaluation subset

Use paired test samples (title + cached image). Cap `N_EVAL` for hosted GPUs; set higher for report-quality curves.


In [ ]:
LABEL_COL = "2_way_label"
TEXT_COL = "clean_title"
ID_COL = "id"
LABEL_NAMES = ["fake", "real"]
N_EVAL = 400  # deletion loops are slow; raise for final report
N_CURVE_EXAMPLES = 80  # examples averaged into deletion curves


def cache_path_for(sample_id: str) -> str:
    return os.path.join(IMAGE_CACHE, f"{str(sample_id).replace('/', '_')}.jpg")


def load_paired(path: str, n=None) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", low_memory=False)
    df[ID_COL] = df[ID_COL].astype(str)
    df["image_path"] = df[ID_COL].map(cache_path_for)
    df = df[df["image_path"].map(os.path.exists)].copy()
    df = df[df[TEXT_COL].notna() & (df[TEXT_COL].astype(str).str.strip() != "")]
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    if n is not None and len(df) > n:
        # stratified-ish
        parts = []
        per = n // 2
        for _, g in df.groupby(LABEL_COL):
            parts.append(g.sample(n=min(len(g), per), random_state=SEED))
        df = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED)
    return df.reset_index(drop=True)


URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")


def clean_text(text) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    return WS_RE.sub(" ", text).strip()


train_df = load_paired(TRAIN_PATH, n=8_000)
test_df = load_paired(TEST_PATH, n=N_EVAL)
test_df["text"] = test_df[TEXT_COL].map(clean_text)
train_df["text"] = train_df[TEXT_COL].map(clean_text)
print(f"train_fit={len(train_df):,}  test_eval={len(test_df):,}")
print(test_df[LABEL_COL].value_counts().sort_index())


## Load frozen models (text / image / fusion)


In [ ]:
# --- DistilBERT (for optional reference probs) ---
tokenizer = AutoTokenizer.from_pretrained(TEXT_CKPT)
text_model = AutoModelForSequenceClassification.from_pretrained(TEXT_CKPT)
text_model.to(DEVICE).eval()
for p in text_model.parameters():
    p.requires_grad = False

# --- ResNet50 ---
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224


def build_resnet50(num_classes: int = 2) -> nn.Module:
    try:
        m = resnet50(weights=None)
    except TypeError:
        m = resnet50(pretrained=False)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


try:
    img_ckpt = torch.load(IMAGE_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    img_ckpt = torch.load(IMAGE_CKPT, map_location="cpu")

image_model = build_resnet50(2)
image_model.load_state_dict(img_ckpt["model_state_dict"])
image_model.to(DEVICE).eval()
for p in image_model.parameters():
    p.requires_grad = False

IMAGE_DIM = 2048
image_encoder = build_resnet50(2)
image_encoder.load_state_dict(img_ckpt["model_state_dict"])
image_encoder.fc = nn.Identity()
image_encoder.to(DEVICE).eval()
for p in image_encoder.parameters():
    p.requires_grad = False

eval_tf = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

# --- Fusion heads (Module 03) ---
class LateFusionMLP(nn.Module):
    def __init__(self, text_dim: int, image_dim: int, hidden: int = 512, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim + image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 2),
        )

    def forward(self, text_emb, image_emb):
        return self.net(torch.cat([text_emb, image_emb], dim=-1))


class CrossAttentionFusion(nn.Module):
    def __init__(self, text_dim: int, image_dim: int, d_model: int = 256, nhead: int = 4, dropout: float = 0.3):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, d_model)
        self.image_proj = nn.Linear(image_dim, d_model)
        self.text_to_image = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.image_to_text = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 4, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 2),
        )

    def forward(self, text_emb, image_emb):
        t = self.text_proj(text_emb).unsqueeze(1)
        i = self.image_proj(image_emb).unsqueeze(1)
        t2i, _ = self.text_to_image(t, i, i, need_weights=False)
        i2t, _ = self.image_to_text(i, t, t, need_weights=False)
        fused = torch.cat([t.squeeze(1), i.squeeze(1), t2i.squeeze(1), i2t.squeeze(1)], dim=-1)
        return self.classifier(fused)


try:
    fusion_blob = torch.load(FUSION_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    fusion_blob = torch.load(FUSION_CKPT, map_location="cpu")

TEXT_DIM = int(fusion_blob.get("text_dim", 768))
late_model = LateFusionMLP(TEXT_DIM, IMAGE_DIM).to(DEVICE)
xattn_model = CrossAttentionFusion(TEXT_DIM, IMAGE_DIM).to(DEVICE)
late_model.load_state_dict(fusion_blob["late_fusion_state_dict"])
xattn_model.load_state_dict(fusion_blob["cross_attn_state_dict"])
late_model.eval()
xattn_model.eval()
for p in list(late_model.parameters()) + list(xattn_model.parameters()):
    p.requires_grad = False

print("Loaded DistilBERT, ResNet50, late fusion, cross-attn fusion.")


## Text faithfulness — SHAP deletion on TF-IDF + LogReg

Fit the Module 01-style linear baseline on the train paired subsample, explain with LinearExplainer, then delete top tokens in importance order and track predicted-class confidence (deletion curve).


In [ ]:
import shap
import inspect

text_baseline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(max_features=40_000, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        ),
        (
            "clf",
            LogisticRegression(max_iter=200, class_weight="balanced", random_state=SEED, n_jobs=-1),
        ),
    ]
)
text_baseline.fit(train_df["text"].tolist(), train_df[LABEL_COL].astype(int).to_numpy())
print("TF-IDF + LogReg fitted for SHAP deletion tests.")

vectorizer = text_baseline.named_steps["tfidf"]
clf = text_baseline.named_steps["clf"]
bg = vectorizer.transform(train_df["text"].tolist()[:200])
params = inspect.signature(shap.LinearExplainer.__init__).parameters
if "feature_perturbation" in params:
    explainer = shap.LinearExplainer(clf, bg, feature_perturbation="interventional")
elif "feature_dependence" in params:
    explainer = shap.LinearExplainer(clf, bg, feature_dependence="independent")
else:
    explainer = shap.LinearExplainer(clf, bg)


def text_pred_proba(text: str) -> np.ndarray:
    return text_baseline.predict_proba([text])[0]


def shap_token_ranking(text: str, toward_class: int):
    """Return tokens sorted by contribution toward `toward_class` (desc)."""
    x = vectorizer.transform([text])
    sv = explainer.shap_values(x)
    if isinstance(sv, list):
        vals = np.asarray(sv[toward_class][0]).ravel()
    else:
        vals = np.asarray(sv[0]).ravel()
        # LinearExplainer often returns values for class 1; flip for class 0
        if toward_class == 0:
            vals = -vals
    feats = np.array(vectorizer.get_feature_names_out())
    # Only keep features that actually appear in this doc
    present = x.toarray()[0] > 0
    idxs = np.where(present)[0]
    if len(idxs) == 0:
        return []
    order = idxs[np.argsort(-vals[idxs])]
    return [(feats[i], float(vals[i])) for i in order]


def mask_tokens(text: str, tokens_to_mask) -> str:
    out = text
    for tok in tokens_to_mask:
        # word-boundary-ish replace for unigrams; soft replace for bigrams
        out = re.sub(rf"\b{re.escape(tok)}\b", " ", out)
        out = out.replace(tok, " ")
    return WS_RE.sub(" ", out).strip()


def text_deletion_curve(text: str, steps: int = 8):
    base = text_pred_proba(text)
    pred = int(np.argmax(base))
    base_conf = float(base[pred])
    ranking = shap_token_ranking(text, toward_class=pred)
    if not ranking:
        return {"pred": pred, "curve": [base_conf], "tokens": []}

    ks = list(range(0, min(steps, len(ranking)) + 1))
    curve = []
    used = []
    for k in ks:
        toks = [t for t, _ in ranking[:k]]
        masked = mask_tokens(text, toks) if k > 0 else text
        if not masked:
            masked = " "
        conf = float(text_pred_proba(masked)[pred])
        curve.append(conf)
        used = toks
    return {"pred": pred, "curve": curve, "tokens": [t for t, _ in ranking[: max(ks)]], "base_conf": base_conf}


# Average deletion curves over a subset (correct + incorrect predictions)
curve_rows = test_df.head(N_CURVE_EXAMPLES)
text_curves = []
text_faith_drops = []  # base_conf - conf_after_top5
for _, row in tqdm(curve_rows.iterrows(), total=len(curve_rows), desc="text_deletion"):
    res = text_deletion_curve(row["text"], steps=8)
    text_curves.append(res["curve"])
    topk = min(5, len(res["curve"]) - 1)
    drop = res["curve"][0] - res["curve"][topk]
    text_faith_drops.append(drop)

# Pad curves to same length
max_len = max(len(c) for c in text_curves)
padded = np.array([c + [c[-1]] * (max_len - len(c)) for c in text_curves], dtype=float)
mean_text_curve = padded.mean(axis=0)
text_faithfulness = float(np.mean(text_faith_drops))  # higher = more faithful
print(f"Mean conf drop after deleting top-5 SHAP tokens: {text_faithfulness:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(range(len(mean_text_curve)), mean_text_curve, marker="o", color="#4cd3c2")
plt.xlabel("# top SHAP tokens removed")
plt.ylabel("Mean confidence in original predicted class")
plt.title("Text deletion curve (TF-IDF + LogReg + SHAP)")
plt.grid(alpha=0.3)
plt.tight_layout()
text_curve_path = os.path.join(OUT_DIR, "text_deletion_curve.png")
plt.savefig(text_curve_path, dpi=150)
plt.show()
print("Wrote", text_curve_path)

# One concrete example
ex = test_df.iloc[0]
ex_res = text_deletion_curve(ex["text"], steps=5)
print("Example text:", ex["text"][:160])
print("Top tokens:", ex_res["tokens"][:5])
print("Confidence curve:", [round(x, 4) for x in ex_res["curve"]])


## Image faithfulness — Grad-CAM deletion

Blank the highest-activation region from Grad-CAM (`layer4[-1]`), re-score ResNet50, and average the confidence drop into a deletion curve (progressive masking of top-% area).


In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except ModuleNotFoundError:
    %pip install -q "grad-cam>=1.5.0"
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


@torch.no_grad()
def image_proba_from_pil(pil_img: Image.Image) -> np.ndarray:
    x = eval_tf(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)
    logits = image_model(x)
    return F.softmax(logits, dim=-1)[0].cpu().numpy()


def get_gradcam_map(pil_img: Image.Image, pred: int) -> np.ndarray:
    target_layers = [image_model.layer4[-1]]
    cam = GradCAM(model=image_model, target_layers=target_layers)
    x = eval_tf(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)
    grayscale = cam(input_tensor=x, targets=[ClassifierOutputTarget(pred)])[0]
    return grayscale  # HxW in [0,1]


def blank_top_cam_region(pil_img: Image.Image, cam_map: np.ndarray, frac: float) -> Image.Image:
    """Set the top-`frac` CAM pixels to black (by area quantile)."""
    img = pil_img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    cam_r = np.array(
        Image.fromarray((cam_map * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))
    ).astype(np.float32) / 255.0
    if frac <= 0:
        return img
    thr = np.quantile(cam_r, 1.0 - frac)
    mask = cam_r >= thr
    arr = np.array(img)
    arr[mask] = 0
    return Image.fromarray(arr)


def image_deletion_curve(path: str, fracs=(0.0, 0.05, 0.1, 0.2, 0.35, 0.5)):
    with Image.open(path) as im:
        pil = im.convert("RGB")
    base = image_proba_from_pil(pil)
    pred = int(np.argmax(base))
    cam_map = get_gradcam_map(pil, pred)
    curve = []
    for f in fracs:
        masked = blank_top_cam_region(pil, cam_map, f)
        conf = float(image_proba_from_pil(masked)[pred])
        curve.append(conf)
    return {"pred": pred, "curve": curve, "fracs": list(fracs), "base_conf": float(base[pred])}


img_rows = curve_rows if "image_path" in curve_rows.columns else test_df.head(N_CURVE_EXAMPLES)
image_curves = []
image_faith_drops = []
FRAC_LIST = [0.0, 0.05, 0.1, 0.2, 0.35, 0.5]

for _, row in tqdm(img_rows.iterrows(), total=len(img_rows), desc="image_deletion"):
    try:
        res = image_deletion_curve(row["image_path"], fracs=FRAC_LIST)
    except Exception:
        continue
    image_curves.append(res["curve"])
    # drop after blanking top 20% CAM area
    idx20 = FRAC_LIST.index(0.2)
    image_faith_drops.append(res["curve"][0] - res["curve"][idx20])

mean_image_curve = np.mean(np.array(image_curves), axis=0)
image_faithfulness = float(np.mean(image_faith_drops)) if image_faith_drops else float("nan")
print(f"Mean conf drop after blanking top-20% Grad-CAM area: {image_faithfulness:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(FRAC_LIST, mean_image_curve, marker="o", color="#ff7a59")
plt.xlabel("Fraction of top Grad-CAM area blanked")
plt.ylabel("Mean confidence in original predicted class")
plt.title("Image deletion curve (ResNet50 + Grad-CAM)")
plt.grid(alpha=0.3)
plt.tight_layout()
img_curve_path = os.path.join(OUT_DIR, "image_deletion_curve.png")
plt.savefig(img_curve_path, dpi=150)
plt.show()
print("Wrote", img_curve_path)


## Missing-modality robustness (fusion)

Zero the text embedding or the image embedding (keep tensor shape), then score late fusion and cross-attn fusion. This matches real text-only / image-only posts better than changing input rank.


In [ ]:
@torch.no_grad()
def encode_text_one(text: str):
    enc = tokenizer(text, truncation=True, max_length=64, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = text_model.distilbert(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    hidden = out.last_hidden_state[:, 0]
    pooled = F.relu(text_model.pre_classifier(hidden))
    logits = text_model.classifier(pooled)
    return pooled.squeeze(0), F.softmax(logits, dim=-1).squeeze(0)


@torch.no_grad()
def encode_image_one(path: str):
    with Image.open(path) as im:
        x = eval_tf(im.convert("RGB")).unsqueeze(0).to(DEVICE)
    feat = image_encoder(x).squeeze(0)
    logits = image_model(x)
    return feat, F.softmax(logits, dim=-1).squeeze(0)


def load_or_build_test_embeddings(df: pd.DataFrame):
    # Prefer Module 03 cache if IDs overlap; else compute for eval subset
    t_path = os.path.join(EMB_CACHE, "test_text.npy")
    i_path = os.path.join(EMB_CACHE, "test_image.npy")
    y_path = os.path.join(EMB_CACHE, "test_labels.npy")
    id_path = os.path.join(EMB_CACHE, "test_ids.npy")
    if all(os.path.exists(p) for p in [t_path, i_path, y_path, id_path]):
        cached_ids = np.load(id_path, allow_pickle=True).astype(str)
        id_to_idx = {i: n for n, i in enumerate(cached_ids)}
        keep = [i for i in df[ID_COL].astype(str) if i in id_to_idx]
        if len(keep) >= max(50, len(df) // 4):
            idxs = [id_to_idx[i] for i in keep]
            print(f"Using Module 03 embedding cache overlap: {len(idxs)}")
            return {
                "text": np.load(t_path)[idxs],
                "image": np.load(i_path)[idxs],
                "y": np.load(y_path)[idxs],
                "ids": cached_ids[idxs],
            }

    texts, images, ys, ids = [], [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="embed_eval"):
        te, _ = encode_text_one(row["text"])
        ie, _ = encode_image_one(row["image_path"])
        texts.append(te.cpu().numpy())
        images.append(ie.cpu().numpy())
        ys.append(int(row[LABEL_COL]))
        ids.append(str(row[ID_COL]))
    return {
        "text": np.stack(texts).astype(np.float32),
        "image": np.stack(images).astype(np.float32),
        "y": np.asarray(ys, dtype=np.int64),
        "ids": np.asarray(ids),
    }


eval_emb = load_or_build_test_embeddings(test_df)


@torch.no_grad()
def fusion_predict(model, text_np, image_np):
    t = torch.from_numpy(text_np).to(DEVICE).float()
    i = torch.from_numpy(image_np).to(DEVICE).float()
    logits = model(t, i)
    return logits.argmax(dim=1).cpu().numpy(), F.softmax(logits, dim=-1).cpu().numpy()


def metrics_from_preds(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    }


ablation_rows = []
for model_name, model in [("Late fusion", late_model), ("Cross-attn fusion", xattn_model)]:
    # both modalities
    pred_both, _ = fusion_predict(model, eval_emb["text"], eval_emb["image"])
    m_both = metrics_from_preds(eval_emb["y"], pred_both)

    # text only → zero image embedding
    zero_img = np.zeros_like(eval_emb["image"])
    pred_t, _ = fusion_predict(model, eval_emb["text"], zero_img)
    m_t = metrics_from_preds(eval_emb["y"], pred_t)

    # image only → zero text embedding
    zero_txt = np.zeros_like(eval_emb["text"])
    pred_i, _ = fusion_predict(model, zero_txt, eval_emb["image"])
    m_i = metrics_from_preds(eval_emb["y"], pred_i)

    ablation_rows.extend(
        [
            {"model": model_name, "setting": "text+image", **m_both},
            {"model": model_name, "setting": "text-only (image=0)", **m_t,
             "delta_acc_vs_both": m_t["accuracy"] - m_both["accuracy"]},
            {"model": model_name, "setting": "image-only (text=0)", **m_i,
             "delta_acc_vs_both": m_i["accuracy"] - m_both["accuracy"]},
        ]
    )
    print(f"\n{model_name}")
    print(f"  both     acc={m_both['accuracy']:.4f}  f1={m_both['f1_macro']:.4f}")
    print(f"  text→0img acc={m_t['accuracy']:.4f}  Δacc={m_t['accuracy']-m_both['accuracy']:+.4f}")
    print(f"  img→0txt  acc={m_i['accuracy']:.4f}  Δacc={m_i['accuracy']-m_both['accuracy']:+.4f}")

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)
ablation_path = os.path.join(OUT_DIR, "missing_modality_ablation.csv")
ablation_df.to_csv(ablation_path, index=False)
print("Wrote", ablation_path)


## Combined comparison table

Pull Module 01–03 metrics when present, attach faithfulness scores from this notebook, and write `module04_metrics.csv`.


In [ ]:
rows = []

# Prior module metrics (Drive copies)
for path, note in [
    (os.path.join(DATA_DIR, "checkpoints", "module01_metrics.csv"), "module01"),
    (os.path.join(DATA_DIR, "checkpoints", "module02_metrics.csv"), "module02"),
    (os.path.join(DATA_DIR, "checkpoints", "module03_metrics.csv"), "module03"),
]:
    if os.path.exists(path):
        tmp = pd.read_csv(path)
        tmp["source"] = note
        rows.append(tmp)
        print(f"Loaded {path}")
    else:
        print(f"Missing optional metrics: {path}")

prior = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

faith_rows = pd.DataFrame(
    [
        {
            "model": "TF-IDF+LogReg (SHAP deletion)",
            "split": "eval_subset",
            "faithfulness_mean_conf_drop": text_faithfulness,
            "faithfulness_protocol": "delete_top5_shap_tokens",
            "n_examples": len(text_faith_drops),
        },
        {
            "model": "ResNet50 (Grad-CAM deletion)",
            "split": "eval_subset",
            "faithfulness_mean_conf_drop": image_faithfulness,
            "faithfulness_protocol": "blank_top20pct_cam",
            "n_examples": len(image_faith_drops),
        },
    ]
)

# Compact leaderboard for the report
leaderboard = []
if len(prior):
    test_prior = prior[prior["split"].astype(str).str.contains("test", case=False)].copy()
    for _, r in test_prior.iterrows():
        leaderboard.append(
            {
                "model": r["model"],
                "split": r["split"],
                "accuracy": r.get("accuracy", np.nan),
                "f1_macro": r.get("f1_macro", np.nan),
                "source": r.get("source", ""),
            }
        )

# Attach ablation both-modality numbers from this run
for _, r in ablation_df[ablation_df["setting"] == "text+image"].iterrows():
    leaderboard.append(
        {
            "model": f"{r['model']} (this eval subset)",
            "split": "eval_subset",
            "accuracy": r["accuracy"],
            "f1_macro": r["f1_macro"],
            "source": "module04_ablation",
        }
    )

leaderboard_df = pd.DataFrame(leaderboard)
display(leaderboard_df)
display(faith_rows)
display(ablation_df)

summary_path = os.path.join(DATA_DIR, "checkpoints", "module04_metrics.csv")
# Store faithfulness + ablation together
out_metrics = {
    "faithfulness": faith_rows.to_dict(orient="records"),
    "missing_modality": ablation_df.to_dict(orient="records"),
    "leaderboard": leaderboard_df.to_dict(orient="records"),
    "text_faithfulness_mean_conf_drop": text_faithfulness,
    "image_faithfulness_mean_conf_drop": image_faithfulness,
}
with open(os.path.join(OUT_DIR, "module04_summary.json"), "w", encoding="utf-8") as f:
    json.dump(out_metrics, f, indent=2)

# Flat CSV for artifacts/ GitHub (faithfulness focus + ablation)
flat = []
for _, r in faith_rows.iterrows():
    flat.append({**r})
for _, r in ablation_df.iterrows():
    flat.append({**r})
flat_df = pd.DataFrame(flat)
flat_df.to_csv(summary_path, index=False)
print("Wrote", summary_path)
print("Wrote", os.path.join(OUT_DIR, "module04_summary.json"))


## Artifacts

| Output | Location |
|--------|----------|
| Text deletion curve | `{DATA_DIR}/checkpoints/module04_evaluation/text_deletion_curve.png` |
| Image deletion curve | `{DATA_DIR}/checkpoints/module04_evaluation/image_deletion_curve.png` |
| Missing-modality table | `{DATA_DIR}/checkpoints/module04_evaluation/missing_modality_ablation.csv` |
| Summary JSON | `{DATA_DIR}/checkpoints/module04_evaluation/module04_summary.json` |
| Metrics CSV | `{DATA_DIR}/checkpoints/module04_metrics.csv` |

Commit `module04_metrics.csv` + the two PNG curves to git under `artifacts/` if you want them in the repo. Large weights stay on Drive.

**Done when:** deletion curves exist, missing-modality deltas are numeric, and the comparison table is filled.

**Next (Module 05):** Streamlit demo — paste text, upload image, show label + confidence + text highlights + Grad-CAM + agreement line.
